**ENTRENAMIENTO DE CLASIFICADOR DE EMOCIONES (HOG + SVM RBF)**

En este cuaderno se desarrolla el entrenamiento y evaluación de un clasificador de emociones faciales basado en la combinación clásica de descriptores de histograma de gradientes orientados junto con una máquina de vectores de soporte con kernel de base radial.

El objetivo es establecer una referencia clásica de rendimiento sobre el conjunto ampliado de expresiones faciales, compensando el desbalance de clases y empaquetando el preprocesado, extracción y clasificación en un único pipeline serializable.

**IMPORTS NECESARIOS Y DEPENDENCIAS**

Se importan los módulos necesarios para la construcción de pipelines y escalado de datos, herramientas de serialización, utilidades de medición temporal y los métodos auxiliares.

In [1]:
import time
import joblib
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from comun import TAM, Extractor, cargar_dataset, informe, registrar_entrenamiento

**CONFIGURACIÓN DE RUTAS Y ARCHIVO DE SALIDA**

Definimos los directorios correspondientes a los conjuntos de datos de entrenamiento, evaluación y salida.

In [2]:
RUTA_TRAIN = "dataset/train"
RUTA_TEST = "dataset/test"
SALIDA = "modelos/hog_svm.joblib"

**CARGA Y NORMALIZACIÓN DEL CONJUNTO DE DATOS**

Se procede a la lectura de las imágenes de entrenamiento y test organizadas por carpetas según su etiqueta emocional. Todas las imágenes son reescaladas a una dimensión normalizada uniforme definida en `TAM`.

In [3]:
# Datos
imgs_tr, ytr = cargar_dataset(RUTA_TRAIN, tam=TAM)
imgs_te, yte = cargar_dataset(RUTA_TEST, tam=TAM)

dataset/train: 30240 imágenes | por clase: {'angry': 3995, 'duckface': 1967, 'fear': 4097, 'happy': 7215, 'neutral': 4965, 'sad': 4830, 'surprise': 3171}
dataset/test: 7396 imágenes | por clase: {'angry': 958, 'duckface': 329, 'fear': 1024, 'happy': 1774, 'neutral': 1233, 'sad': 1247, 'surprise': 831}


**EXTRACCIÓN DE CARACTERÍSTICAS HOG**

Utilizando el método `hog`, se calcula el vector de descriptores de gradientes para cada imagen.

In [4]:
# Extracción de características
extractor = Extractor("hog")
Xtr = extractor.transform(imgs_tr)
Xte = extractor.transform(imgs_te)

**DEFINICIÓN DEL PIPELINE CLASIFICADOR (STANDARDSCALER + SVM RBF)**

Se construye un pipeline de clasificación compuesto por una normalización estándar de características y una máquina de vectores de soporte con kernel RBF. Se utiliza `class_weight='balanced'` para ajustar automáticamente los pesos de la función de coste y compensar el menor número de muestras de las clases minoritarias, como `duckface`.

In [5]:
# Clasificador
clasificador = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf", C=5, class_weight="balanced"),
)

**BÚSQUEDA DE HIPERPARÁMETROS**

Para justificar la elección de `C` y `gamma` se hace una búsqueda en rejilla con validación cruzada estratificada de 3 particiones, optimizando el **F1 macro**. Para que sea viable en CPU se usa un subconjunto estratificado de 6.000 muestras de entrenamiento; el test no se toca.

In [6]:
BUSCAR_HIPERPARAMETROS = False      # True: tarda varios minutos

if BUSCAR_HIPERPARAMETROS:
    import pandas as pd
    from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
    Xs, _, ys, _ = train_test_split(Xtr, ytr, train_size=6000, stratify=ytr, random_state=42)
    rejilla = GridSearchCV(
        clasificador,
        {"svc__C": [1, 5, 10], "svc__gamma": [1e-4, "scale", 2e-3]},
        cv=StratifiedKFold(3, shuffle=True, random_state=42),
        scoring="f1_macro", n_jobs=-1, verbose=1)
    rejilla.fit(Xs, ys)
    res = pd.DataFrame(rejilla.cv_results_)[["param_svc__C", "param_svc__gamma", "mean_test_score", "std_test_score", "mean_fit_time"]]
    display(res.sort_values("mean_test_score", ascending=False).round(4))
    print("Mejores parámetros:", rejilla.best_params_, f"| F1 macro (CV) = {rejilla.best_score_:.4f}")
    clasificador.set_params(**rejilla.best_params_)

**ENTRENAMIENTO DEL MODELO**

Se ajusta el clasificador sobre el conjunto de entrenamiento de descriptores HOG y sus correspondientes etiquetas, monitorizando el tiempo total requerido por la fase de optimización.

In [7]:
t0 = time.time()
clasificador.fit(Xtr, ytr)
t_entreno = time.time() - t0

**EVALUACIÓN CUANTITATIVA SOBRE EL CONJUNTO DE TEST**

Se realiza la predicción sobre las muestras de test no vistas durante el ajuste y se genera el informe, reportando métricas de precisión, recall, F1-score por clase y precisión global, además del tiempo medio de inferencia.

In [8]:
# Evaluación
t0 = time.time()
pred = clasificador.predict(Xte)
t_pred = time.time() - t0
metricas = informe(yte, pred, t_entreno, t_pred)

              precision    recall  f1-score   support

       angry       0.45      0.40      0.43       958
    duckface       0.67      0.85      0.75       329
        fear       0.46      0.39      0.42      1024
       happy       0.67      0.78      0.72      1774
     neutral       0.51      0.49      0.50      1233
         sad       0.43      0.46      0.44      1247
    surprise       0.77      0.64      0.70       831

    accuracy                           0.56      7396
   macro avg       0.56      0.57      0.56      7396
weighted avg       0.56      0.56      0.56      7396

Matriz de confusión (filas = real, columnas = predicho):
[[ 387   19  105  142  119  173   13]
 [   2  279    6   13    4   24    1]
 [ 113   15  395  101  127  200   73]
 [  75   38   56 1379   86  111   29]
 [ 100   25   96  186  606  195   25]
 [ 133   29  127  178  192  568   20]
 [  45   14   71   65   59   47  530]]

Accuracy test: 0.5603 | F1 macro: 0.5644
Tiempo de entrenamiento: 3111.2 s
Inf

**SERIALIZACIÓN Y GUARDADO DEL PIPELINE COMPLETO**

Se encapsulan tanto la fase de extracción de características como el modelo clasificador en un único `Pipeline`.

In [9]:
# Guardado
modelo = Pipeline([("extractor", extractor), ("clasificador", clasificador)])
joblib.dump(modelo, SALIDA)
print("Modelo guardado en", SALIDA)
registrar_entrenamiento(SALIDA, t_entreno, ms_imagen=1000 * t_pred / len(yte), **metricas)

Modelo guardado en modelos/hog_svm.joblib
Registrado en informe_entrenamiento\registro_modelos.json
